In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")

print("Path to dataset files:", path)

In [2]:
import pandas as pd

df = pd.read_csv(path + "/AB_NYC_2019.csv")

In [3]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  str    
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  str    
 4   neighbourhood_group             48895 non-null  str    
 5   neighbourhood                   48895 non-null  str    
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  str    
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     38843 non-n

### not useful :
id , host_id , host name during training . 

In [5]:
df.isnull().sum()

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [16]:
(df.last_review.isnull().sum() / len(df)) * 100

np.float64(20.55833929849678)

In [48]:
# drop id , host_id , host_name , 
# and create a new df wihtout these columns 
df1 = df.drop(['id', 'host_name'], axis=1)
# replace the null values of name column with 'unknown' 
df1['name'] = df1['name'].fillna('unknown')
# dropping the last_review column as it seems to hold no value :
df1 = df1.drop(['reviews_per_month'], axis=1)
# finding the price of the listing per night 
df1['price_per_night'] = df1['price'] / (df1['minimum_nights'])
# calculating the time when the reviews were made 
# and creating a new column with the time difference in days
reference_date = df1["last_review"].astype('datetime64[ns]').max()
days_since_last_review = (reference_date - df1["last_review"].astype('datetime64[ns]')).dt.days
df1['days_since_last_review'] = days_since_last_review
# the 0 reviews mean that no reviews were ever made for the listing, 
# so we can replace the null values with 0
df1.fillna({col : 0  for col in df1.columns if col == 'days_since_last_review'}, inplace=True)
# dropping the last_review column as it seems to hold no value :
df1 = df1.drop(['last_review'], axis=1)

In [51]:
# what all columns have been added and what all columns have been dropped
print("Columns added:")
for col in df1.columns:
    if col not in df.columns:
        print(col)
print("\nColumns dropped:")
for col in df.columns:
    if col not in df1.columns:
        print(col)

Columns added:
price_per_night
days_since_last_review

Columns dropped:
id
host_name
last_review
reviews_per_month


In [49]:
df1.head()

,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,price_per_night,days_since_last_review
0,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,6,365,149.0,262.0
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2,355,225.0,48.0
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,1,365,50.0,0.0
3,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,1,194,89.0,3.0
4,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,1,0,8.0,231.0


In [52]:
df1.isnull().sum()

name                              0
host_id                           0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
calculated_host_listings_count    0
availability_365                  0
price_per_night                   0
days_since_last_review            0
dtype: int64

In [54]:
# converting thsi dataset to .csv file to work on in another .ipynb file
df1.to_csv('AB_NYC_2019_cleaned.csv', index=False)